# War Supply Problem with AMPL in Python
## AMPL Modeling for Book Problem 3.1

This notebook models the war-supply mission from book section `3.1` with AMPL from Python using `amplpy`.

The textbook reports a minimum mission cost of `4,820,000` USD. Because the model has multiple optimal flight plans, this notebook validates the objective value and feasibility rather than forcing one specific distribution of flights.


## Requirements

This notebook uses `amplpy` together with the HiGHS solver module.

If your environment does not have `amplpy`, install it first:

```python
%pip install amplpy
```

The first call to `ampl_notebook(modules=["highs"], license_uuid="default")` may download the solver module if it is not already available.


In [1]:
from __future__ import annotations

from functools import wraps
from math import isclose
from time import perf_counter


In [2]:
%pip install amplpy
try:
    from amplpy import ampl_notebook
except ImportError as exc:
    raise ImportError(
        "This notebook requires amplpy. Install it with `%pip install amplpy` before running."
    ) from exc


def timed(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        result = func(*args, **kwargs)
        elapsed = perf_counter() - start
        return result, elapsed

    return wrapper


def create_ampl_instance(solver: str = "highs"):
    ampl = ampl_notebook(modules=[solver], license_uuid="default")
    ampl.option["solver"] = solver
    ampl.option["solver_msg"] = 0
    return ampl


def round_if_close(value: float, digits: int = 4):
    rounded = round(value, digits)
    return int(round(rounded)) if isclose(rounded, round(rounded), abs_tol=1e-9) else rounded


def extract_2d_solution(variable, row_keys, col_keys):
    raw = variable.get_values().to_dict()
    values = {}
    for row in row_keys:
        for col in col_keys:
            values[(row, col)] = round_if_close(float(raw[(row, col)]))
    return values


Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: /Applications/Xcode.app/Contents/Developer/usr/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Problem: War Supply Mission

Two aircraft types must supply strategic points `A` and `B`.

Point `A` needs minimum armament, vehicles and jeeps. Point `B` needs minimum armament and must respect a fuel-capacity combination expressed through vehicles and jeeps. The clandestine airport also limits the combined aircraft usage and the available stock of all transported items.


In [3]:
AIRCRAFT = ["tip1", "tip2"]
DESTINATIONS = ["A", "B"]
ITEMS = ["armament", "vehicles", "jeeps"]

COST = {
    ("tip1", "A"): 120_000,
    ("tip1", "B"): 80_000,
    ("tip2", "A"): 90_000,
    ("tip2", "B"): 60_000,
}
CAPACITY = {
    ("tip1", "armament"): 20,
    ("tip1", "vehicles"): 4,
    ("tip1", "jeeps"): 3,
    ("tip2", "armament"): 15,
    ("tip2", "vehicles"): 1,
    ("tip2", "jeeps"): 2,
}
DEMAND_A = {"armament": 570, "vehicles": 25, "jeeps": 18}
DEMAND_B_ARMAMENT = 350
AIRPORT_STOCK = {"armament": 2_000, "vehicles": 150, "jeeps": 230}
EXPECTED_OBJECTIVE = 4_820_000


def summarize_solution(flights):
    deliveries = {destination: {item: 0 for item in ITEMS} for destination in DESTINATIONS}
    for aircraft in AIRCRAFT:
        for destination in DESTINATIONS:
            flown = flights[(aircraft, destination)]
            for item in ITEMS:
                deliveries[destination][item] += CAPACITY[(aircraft, item)] * flown
    airport_usage = {item: deliveries["A"][item] + deliveries["B"][item] for item in ITEMS}
    return deliveries, airport_usage


def solution_is_feasible(result):
    deliveries, airport_usage = summarize_solution(result["flights"])
    if any(deliveries["A"][item] < DEMAND_A[item] for item in ITEMS):
        return False
    if deliveries["B"]["armament"] < DEMAND_B_ARMAMENT:
        return False
    if deliveries["B"]["vehicles"] / 120 + deliveries["B"]["jeeps"] / 90 > 1 + 1e-9:
        return False
    if result["flights"][("tip1", "A")] / 120 + result["flights"][("tip1", "B")] / 120 + result["flights"][("tip2", "A")] / 85 + result["flights"][("tip2", "B")] / 85 > 1 + 1e-9:
        return False
    if any(airport_usage[item] > AIRPORT_STOCK[item] for item in ITEMS):
        return False
    return True


@timed
def solve_war_supply_with_ampl(solver: str = "highs"):
    ampl = create_ampl_instance(solver)
    ampl.eval(
        r'''
        set AIRCRAFT ordered;
        set DESTINATIONS ordered;
        set ITEMS ordered;

        param cost {AIRCRAFT, DESTINATIONS} >= 0;
        param capacity {AIRCRAFT, ITEMS} >= 0;
        param demand_a {ITEMS} >= 0;
        param demand_b_armament >= 0;
        param airport_stock {ITEMS} >= 0;

        var Flights {a in AIRCRAFT, d in DESTINATIONS} integer >= 0;

        minimize TotalCost:
            sum {a in AIRCRAFT, d in DESTINATIONS} cost[a, d] * Flights[a, d];

        subject to DemandA {i in ITEMS}:
            sum {a in AIRCRAFT} capacity[a, i] * Flights[a, "A"] >= demand_a[i];

        subject to DemandBArmament:
            sum {a in AIRCRAFT} capacity[a, "armament"] * Flights[a, "B"] >= demand_b_armament;

        subject to FuelAtB:
            sum {a in AIRCRAFT} capacity[a, "vehicles"] * Flights[a, "B"] / 120
            + sum {a in AIRCRAFT} capacity[a, "jeeps"] * Flights[a, "B"] / 90 <= 1;

        subject to AirportTime:
            sum {d in DESTINATIONS} Flights["tip1", d] / 120
            + sum {d in DESTINATIONS} Flights["tip2", d] / 85 <= 1;

        subject to AirportStock {i in ITEMS}:
            sum {a in AIRCRAFT, d in DESTINATIONS} capacity[a, i] * Flights[a, d] <= airport_stock[i];
        '''
    )
    ampl.set["AIRCRAFT"] = AIRCRAFT
    ampl.set["DESTINATIONS"] = DESTINATIONS
    ampl.set["ITEMS"] = ITEMS
    ampl.param["cost"] = COST
    ampl.param["capacity"] = CAPACITY
    ampl.param["demand_a"] = DEMAND_A
    ampl.param["demand_b_armament"] = DEMAND_B_ARMAMENT
    ampl.param["airport_stock"] = AIRPORT_STOCK
    ampl.solve()

    flights = extract_2d_solution(ampl.var["Flights"], AIRCRAFT, DESTINATIONS)
    deliveries, airport_usage = summarize_solution(flights)
    return {
        "flights": flights,
        "deliveries": deliveries,
        "airport_usage": airport_usage,
        "cost": round_if_close(ampl.obj["TotalCost"].value()),
    }


ampl_result, ampl_time = solve_war_supply_with_ampl()
print("=== WAR SUPPLY RESULTS WITH AMPL ===")
print(f"Solution -> {ampl_result}")
print(f"Time     -> {ampl_time:.8f} seconds")

assert ampl_result["cost"] == EXPECTED_OBJECTIVE
assert solution_is_feasible(ampl_result)
print("AMPL reaches the published minimum cost and returns a feasible optimal flight plan.")


/Users/lfuentesl/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


AMPL Version 20250901 (Darwin-22.6.0, 64-bit)
Demo license with maintenance expiring 20270131.
Using license file "/Users/lfuentesl/Library/Python/3.9/lib/python/site-packages/ampl_module_base/bin/ampl.lic".

HiGHS 1.11.0: === WAR SUPPLY RESULTS WITH AMPL ===
Solution -> {'flights': {('tip1', 'A'): 18, ('tip1', 'B'): 4, ('tip2', 'A'): 14, ('tip2', 'B'): 18}, 'deliveries': {'A': {'armament': 570, 'vehicles': 86, 'jeeps': 82}, 'B': {'armament': 350, 'vehicles': 34, 'jeeps': 48}}, 'airport_usage': {'armament': 920, 'vehicles': 120, 'jeeps': 130}, 'cost': 4820000}
Time     -> 1.68422233 seconds
AMPL reaches the published minimum cost and returns a feasible optimal flight plan.


## Note on the Published Plan

The book prints the plan `(21, 10)` flights to point `A` and `(10, 10)` flights to point `B`. Since the model has multiple optima, a solver may return another feasible optimal plan with the same minimum cost.
